# Segment Wise Modelling

This notebook runs segment-wise tea price prediction on `tea_preprocessed_v2.csv` (3,034 rows) for four segments: High Grown, Low Grown, Off-Grade, and Dust.

It uses raw target price (`price_mid_lkr`) and evaluates XGBoost, LightGBM, Gradient Boosting, and Random Forest with mandatory 5-fold time-series cross-validation to avoid data leakage.

Features include engineered columns from `feature_engineering.py`, lagged weather variables (1/2/3-week lags for precipitation, sunshine, and temperature), and structural features (`grade_rank`, `tier_rank`, `prestige_rank`).

In [ ]:
"""Imports and global config for segment-wise time-series modeling."""

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import TimeSeriesSplit
from sklearn.pipeline import Pipeline

SEED = 42
np.random.seed(SEED)

try:
    from xgboost import XGBRegressor
except ImportError as exc:
    raise ImportError("Please install xgboost: pip install xgboost") from exc

try:
    from lightgbm import LGBMRegressor
except ImportError as exc:
    raise ImportError("Please install lightgbm: pip install lightgbm") from exc

print("Libraries loaded.")

In [8]:
"""Load data, validate engineered input, and prepare segment dataframes."""

df_raw = pd.read_csv("../data/Processed/tea_preprocessed_v2.csv")
TARGET = "price_mid_lkr"
SEGMENT_COL = "category_type"

# Keep the original row count for traceability to source dataset size.
raw_rows = len(df_raw)
df = df_raw.copy()

# Basic cleaning for robust model training.
df = df.replace([np.inf, -np.inf], np.nan)
df = df.dropna(subset=[TARGET, SEGMENT_COL]).copy()

# Validate that feature_engineering.py outputs are present in this input.
fe_interact = [c for c in df.columns if c.startswith("interact__")]
fe_roll = [c for c in df.columns if c.startswith("roll3_")]
fe_poly = [c for c in df.columns if c.startswith("poly2__")]
if not (fe_interact and fe_roll and fe_poly):
    raise ValueError(
        "Feature-engineered columns not found. Please generate tea_preprocessed_v2.csv using feature_engineering.py."
    )

# Structural features requested in the methodology.
grade_rank_map = {
    "bop": 4, "bopf": 4, "bop1": 4,
    "fbop": 3, "fbop1": 3, "fbopf": 3, "fbopf1": 3, "fbopf_tippy": 3,
    "op": 2, "op1": 2, "opa": 2,
    "pekoe": 1, "pek1": 1, "pekoe_fbop": 1,
}
tier_rank_map = {"select_best": 4, "best": 3, "below_best": 2, "others": 1}
prestige_rank_map = {"high": 3, "medium": 2, "low": 1}

df["grade_rank"] = df.get("grade", pd.Series("", index=df.index)).astype(str).str.lower().map(grade_rank_map).fillna(0)
df["tier_rank"] = df.get("tier", pd.Series("", index=df.index)).astype(str).str.lower().map(tier_rank_map).fillna(df.get("tier_enc", 0)).fillna(0)
df["prestige_rank"] = df.get("elevation", pd.Series("", index=df.index)).astype(str).str.lower().map(prestige_rank_map).fillna(0)

# Required lagged weather variables: lag1/lag2/lag3 for precipitation, sunshine, temperature.
lag_weather_cols = [
    c for c in df.columns
    if ("lag1" in c or "lag2" in c or "lag3" in c)
    and ("precipitation" in c or "sunshine" in c or "temperature" in c)
]

# Remove target-derived and same-auction price proxy columns to prevent leakage.
drop_cols = {
    TARGET,
    "price_mid_lkr_log",
    "price_mid_usd",
    "price_lo_lkr",
    "price_hi_lkr",
    "price_range_lkr",
    "sale_id",
    "sale_date_raw",
    "has_price_target",
}
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
candidate_features = [c for c in numeric_cols if c not in drop_cols]

# Force required methodological features into feature set if present.
required_structural = ["grade_rank", "tier_rank", "prestige_rank"]
required_features = [c for c in (lag_weather_cols + required_structural) if c in df.columns]
feature_cols = sorted(list(set(candidate_features + required_features)))

# Create 4 required segment dataframes.
segments = {
    "high_grown": df[df[SEGMENT_COL] == "high_grown"].copy(),
    "low_grown": df[df[SEGMENT_COL] == "low_grown"].copy(),
    "off_grade": df[df[SEGMENT_COL] == "off_grade"].copy(),
    "dust": df[df[SEGMENT_COL] == "dust"].copy(),
}

segment_labels = {
    "high_grown": "High Grown",
    "low_grown": "Low Grown",
    "off_grade": "Off-Grade",
    "dust": "Dust",
}

print(f"Rows loaded from source: {raw_rows}")
print(f"Rows used after target cleanup: {len(df)}")
print(f"Engineered feature counts -> interact: {len(fe_interact)}, roll3: {len(fe_roll)}, poly2: {len(fe_poly)}")
print(f"Lag-weather features found: {len(lag_weather_cols)}")
print(f"Total modeling features (leakage-safe): {len(feature_cols)}")
for key, sdf in segments.items():
    print(f"{segment_labels[key]:<10s}: {len(sdf)} rows")

Rows loaded from source: 3034
Rows used after target cleanup: 2886
Engineered feature counts -> interact: 28, roll3: 19, poly2: 15
Lag-weather features found: 36
Total modeling features (leakage-safe): 231
High Grown: 516 rows
Low Grown : 1348 rows
Off-Grade : 499 rows
Dust      : 523 rows


In [9]:
"""Define models and 5-fold time-series CV evaluation per segment."""

models = {
    "XGBoost": XGBRegressor(
        n_estimators=250,
        learning_rate=0.05,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=SEED,
        n_jobs=-1,
    ),
    "LightGBM": LGBMRegressor(
        n_estimators=250,
        learning_rate=0.05,
        max_depth=-1,
        random_state=SEED,
        n_jobs=-1,
        verbosity=-1,
    ),
    "Gradient Boosting": GradientBoostingRegressor(
        n_estimators=250,
        learning_rate=0.05,
        max_depth=3,
        random_state=SEED,
    ),
    "Random Forest": RandomForestRegressor(
        n_estimators=300,
        random_state=SEED,
        n_jobs=-1,
    ),
}

def compute_metrics(y_true, y_pred):
    """Compute RMSE, MAE, MAPE, and R2 on raw LKR prices."""
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    mape = np.mean(np.abs((y_true - y_pred) / np.maximum(np.abs(y_true), 1e-9))) * 100
    r2 = r2_score(y_true, y_pred)
    return rmse, mae, mape, r2

def evaluate_segment(segment_df, segment_name):
    """Run 5-fold TimeSeriesSplit for all models on one segment."""
    # Sort by auction order so each fold trains on past and tests on future only.
    sort_cols = [c for c in ["sale_year", "sale_number"] if c in segment_df.columns]
    if sort_cols:
        segment_df = segment_df.sort_values(sort_cols).reset_index(drop=True)

    X_seg = segment_df[feature_cols].copy()
    y_seg = segment_df[TARGET].copy()

    # Fold-safe imputation happens inside each fold through a pipeline.
    tscv = TimeSeriesSplit(n_splits=5)
    fold_rows = []

    for fold_idx, (train_idx, test_idx) in enumerate(tscv.split(X_seg), start=1):
        X_train, X_test = X_seg.iloc[train_idx], X_seg.iloc[test_idx]
        y_train, y_test = y_seg.iloc[train_idx], y_seg.iloc[test_idx]

        for model_name, model in models.items():
            pipe = Pipeline([
                ("impute", SimpleImputer(strategy="median")),
                ("model", clone(model)),
            ])
            pipe.fit(X_train, y_train)
            preds = pipe.predict(X_test)
            rmse, mae, mape, r2 = compute_metrics(y_test, preds)

            fold_rows.append(
                {
                    "Segment": segment_name,
                    "Fold": fold_idx,
                    "Model": model_name,
                    "RMSE": rmse,
                    "MAE": mae,
                    "MAPE": mape,
                    "R2": r2,
                    "n_train": len(train_idx),
                    "n_test": len(test_idx),
                }
            )

    fold_results = pd.DataFrame(fold_rows)
    summary = (
        fold_results.groupby(["Segment", "Model"], as_index=False)
        .agg(
            RMSE_mean=("RMSE", "mean"), RMSE_std=("RMSE", "std"),
            MAE_mean=("MAE", "mean"), MAE_std=("MAE", "std"),
            MAPE_mean=("MAPE", "mean"), MAPE_std=("MAPE", "std"),
            R2_mean=("R2", "mean"), R2_std=("R2", "std"),
        )
        .sort_values(["Segment", "RMSE_mean"])
        .reset_index(drop=True)
    )
    return fold_results, summary

In [10]:
"""Run segment-wise CV for all segments and show results."""

all_fold_results = []
all_summary_results = []

for seg_key in ["high_grown", "low_grown", "off_grade", "dust"]:
    seg_df = segments[seg_key]
    seg_name = segment_labels[seg_key]

    if len(seg_df) < 30:
        print(f"Skipping {seg_name}: not enough rows for stable 5-fold CV.")
        continue

    fold_df, summary_df = evaluate_segment(seg_df, seg_name)
    all_fold_results.append(fold_df)
    all_summary_results.append(summary_df)
    print(f"Completed: {seg_name}")

cv_fold_results = pd.concat(all_fold_results, ignore_index=True)
cv_summary_results = pd.concat(all_summary_results, ignore_index=True)

print("\nSegment-wise 5-fold TimeSeries CV summary (sorted by RMSE):")
display(cv_summary_results.round(4))

print("\nBest model per segment (lowest RMSE_mean):")
best_per_segment = (
    cv_summary_results.sort_values(["Segment", "RMSE_mean"])
    .groupby("Segment", as_index=False)
    .first()[["Segment", "Model", "RMSE_mean", "MAE_mean", "MAPE_mean", "R2_mean"]]
    .sort_values("Segment")
    .reset_index(drop=True)
    .round(4)
)
display(best_per_segment)

Completed: High Grown
Completed: Low Grown
Completed: Off-Grade
Completed: Dust

Segment-wise 5-fold TimeSeries CV summary (sorted by RMSE):


,Segment,Model,RMSE_mean,RMSE_std,MAE_mean,MAE_std,MAPE_mean,MAPE_std,R2_mean,R2_std
0,High Grown,Random Forest,224.7682,28.2667,156.6436,23.2764,18.9225,3.2855,0.2871,0.0860
1,High Grown,LightGBM,229.9959,34.0834,165.4046,26.3468,19.8451,3.7087,0.2588,0.0580
2,High Grown,XGBoost,234.0763,34.7721,162.9035,31.5620,19.8750,4.1395,0.2313,0.0838
3,High Grown,Gradient Boosting,238.4481,29.0502,168.1638,29.0486,20.1998,3.9113,0.1947,0.1222
4,Low Grown,Gradient Boosting,280.5365,62.1155,153.7782,19.8129,8.9725,1.0304,0.8581,0.0654
5,Low Grown,Random Forest,285.7480,81.3579,155.7914,32.4983,9.0175,1.6040,0.8469,0.0913
6,Low Grown,LightGBM,344.0631,135.2432,183.3618,56.0898,10.7604,2.6002,0.7607,0.1795
7,Low Grown,XGBoost,354.6926,105.0049,173.2753,36.6366,9.2957,0.9450,0.7649,0.1256
8,Off-Grade,XGBoost,145.1082,25.1306,106.9364,15.9521,15.4117,3.4502,0.2832,0.2275
9,Off-Grade,Random Forest,145.7976,26.7423,109.4378,16.7224,15.7240,3.8444,0.2737,0.2549



Best model per segment (lowest RMSE_mean):


,Segment,Model,RMSE_mean,MAE_mean,MAPE_mean,R2_mean
0,Dust,Random Forest,157.9989,109.9174,12.9115,0.4151
1,High Grown,Random Forest,224.7682,156.6436,18.9225,0.2871
2,Low Grown,Gradient Boosting,280.5365,153.7782,8.9725,0.8581
3,Off-Grade,XGBoost,145.1082,106.9364,15.4117,0.2832


## Hyperparameter Tuning

This section applies the following hyperparameter tuning pipeline:
- 5-fold TimeSeriesSplit
- GridSearchCV
- tuning is done separately for each segment and for each model

In [13]:
"""Run segment-wise GridSearchCV for all models (no best-model filtering)."""

inner_cv = TimeSeriesSplit(n_splits=5)

all_tuning_logs = []
all_tuned_metrics = []

for seg_name, sdf in segments.items():
    print(f"\nTuning segment: {seg_name} | rows={len(sdf)}")

    if len(sdf) < 20:
        print("  Skipped: not enough rows for stable 5-fold tuning.")
        continue

    X_seg = sdf[feature_cols].copy()
    y_seg = sdf[TARGET].copy()

    for model_name, base_model in models.items():
        if model_name not in PARAM_GRIDS_ALL_MODELS:
            print(f"  Skipped {model_name}: no parameter grid found.")
            continue

        estimator = build_tuning_estimator(model_name, base_model)
        param_grid = PARAM_GRIDS_ALL_MODELS[model_name]

        gscv = GridSearchCV(
            estimator=estimator,
            param_grid=param_grid,
            scoring="neg_root_mean_squared_error",
            cv=inner_cv,
            n_jobs=-1,
            refit=True,
            verbose=0,
            return_train_score=False,
        )
        gscv.fit(X_seg, y_seg)

        # Keep full grid search logs for each segment and model.
        grid_log = pd.DataFrame(gscv.cv_results_)[
            ["params", "mean_test_score", "std_test_score", "rank_test_score"]
        ].copy()
        grid_log["Segment"] = seg_name
        grid_log["Model"] = model_name
        grid_log["mean_test_RMSE"] = -grid_log["mean_test_score"]
        all_tuning_logs.append(grid_log)

        # Evaluate tuned estimator again with 5-fold time-series splits.
        tuned_fold_eval = evaluate_tuned_estimator_timeseries(
            sdf=sdf,
            estimator=gscv.best_estimator_,
            n_splits=5,
        )

        all_tuned_metrics.append({
            "Segment": seg_name,
            "Model": model_name,
            "RMSE_mean": tuned_fold_eval["RMSE"].mean(),
            "RMSE_std": tuned_fold_eval["RMSE"].std(),
            "MAE_mean": tuned_fold_eval["MAE"].mean(),
            "MAE_std": tuned_fold_eval["MAE"].std(),
            "MAPE_mean": tuned_fold_eval["MAPE"].mean(),
            "MAPE_std": tuned_fold_eval["MAPE"].std(),
            "R2_mean": tuned_fold_eval["R2"].mean(),
            "R2_std": tuned_fold_eval["R2"].std(),
            "Selected_Hyperparameters": json.dumps(gscv.best_params_, sort_keys=True),
        })

        print(f"  Done: {model_name} | tuned RMSE={tuned_fold_eval['RMSE'].mean():.2f}")

segmentwise_tuning_logs = pd.concat(all_tuning_logs, ignore_index=True) if all_tuning_logs else pd.DataFrame()
segmentwise_tuned_summary = pd.DataFrame(all_tuned_metrics)

print("\nTuning logs shape:", segmentwise_tuning_logs.shape)
print("Tuned summary shape:", segmentwise_tuned_summary.shape)

display(
    segmentwise_tuned_summary
    .sort_values(["Segment", "RMSE_mean"])
    .reset_index(drop=True)
    .round(4)
)


Tuning segment: high_grown | rows=516
  Done: XGBoost | tuned RMSE=232.75
  Done: LightGBM | tuned RMSE=227.74
  Done: Gradient Boosting | tuned RMSE=232.38
  Done: Random Forest | tuned RMSE=225.07

Tuning segment: low_grown | rows=1348
  Done: XGBoost | tuned RMSE=408.81
  Done: LightGBM | tuned RMSE=331.82
  Done: Gradient Boosting | tuned RMSE=263.34
  Done: Random Forest | tuned RMSE=283.19

Tuning segment: off_grade | rows=499
  Done: XGBoost | tuned RMSE=147.05
  Done: LightGBM | tuned RMSE=150.42
  Done: Gradient Boosting | tuned RMSE=156.53
  Done: Random Forest | tuned RMSE=145.11

Tuning segment: dust | rows=523
  Done: XGBoost | tuned RMSE=155.89
  Done: LightGBM | tuned RMSE=161.14
  Done: Gradient Boosting | tuned RMSE=166.36
  Done: Random Forest | tuned RMSE=158.30

Tuning logs shape: (420, 7)
Tuned summary shape: (16, 11)


,Segment,Model,RMSE_mean,RMSE_std,MAE_mean,MAE_std,MAPE_mean,MAPE_std,R2_mean,R2_std,Selected_Hyperparameters
0,dust,XGBoost,155.8868,20.5053,108.8595,13.5598,12.9787,1.9396,0.4273,0.1787,"{""model__learning_rate"": 0.05, ""model__max_dep..."
1,dust,Random Forest,158.3003,15.1429,109.5878,10.9465,12.8837,1.4685,0.4131,0.1541,"{""model__max_depth"": 16, ""model__min_samples_l..."
2,dust,LightGBM,161.1379,20.8432,115.5919,16.5197,13.6848,2.1962,0.3913,0.1763,"{""model__learning_rate"": 0.03, ""model__n_estim..."
3,dust,Gradient Boosting,166.3600,21.6791,117.2099,10.0930,13.9177,1.6818,0.3513,0.1854,"{""model__learning_rate"": 0.03, ""model__max_dep..."
4,high_grown,Random Forest,225.0651,29.1555,155.6405,23.3280,18.9935,3.4999,0.2867,0.0735,"{""model__max_depth"": 8, ""model__min_samples_le..."
5,high_grown,LightGBM,227.7371,33.3049,161.4617,25.9648,19.6037,3.7562,0.2737,0.0459,"{""model__learning_rate"": 0.03, ""model__n_estim..."
6,high_grown,Gradient Boosting,232.3843,35.0363,161.4406,27.6186,19.9873,4.4777,0.2447,0.0487,"{""model__learning_rate"": 0.03, ""model__max_dep..."
7,high_grown,XGBoost,232.7533,32.6161,162.5668,28.3989,19.8182,3.9574,0.2402,0.0579,"{""model__learning_rate"": 0.03, ""model__max_dep..."
8,low_grown,Gradient Boosting,263.3410,54.3416,140.1033,19.0782,8.1742,0.8848,0.8760,0.0503,"{""model__learning_rate"": 0.05, ""model__max_dep..."
9,low_grown,Random Forest,283.1922,80.7930,154.1096,32.4372,8.9281,1.6093,0.8496,0.0897,"{""model__max_depth"": 16, ""model__min_samples_l..."


In [ ]:
import os
import numpy as np
import pandas as pd

# Allow running this cell directly by loading prior outputs from disk when needed.
summary_csv = "../reports/tables/segmentwise_hyperparam_summary_all_models.csv"
logs_csv = "../reports/tables/_intermediate/segmentwise_hyperparam_gridsearch_all_results.csv"

if "segmentwise_tuned_summary" not in globals() or segmentwise_tuned_summary is None or segmentwise_tuned_summary.empty:
    if not os.path.exists(summary_csv):
        raise FileNotFoundError(
            f"Missing {summary_csv}. Run the tuning/save cells first or place the file in reports/tables."
        )
    segmentwise_tuned_summary = pd.read_csv(summary_csv)
    print(f"Loaded segmentwise_tuned_summary from: {summary_csv}")

if "segmentwise_tuning_logs" not in globals() or segmentwise_tuning_logs is None or segmentwise_tuning_logs.empty:
    if not os.path.exists(logs_csv):
        raise FileNotFoundError(
            f"Missing {logs_csv}. Run the tuning/save cells first or place the file in reports/tables."
        )
    segmentwise_tuning_logs = pd.read_csv(logs_csv)
    print(f"Loaded segmentwise_tuning_logs from: {logs_csv}")

# 1) Detailed model comparison table (all segment-model rows).
performance_table = segmentwise_tuned_summary.copy()
performance_table["RMSE_rank_in_segment"] = performance_table.groupby("Segment")["RMSE_mean"].rank(method="dense", ascending=True)
performance_table["R2_rank_in_segment"] = performance_table.groupby("Segment")["R2_mean"].rank(method="dense", ascending=False)
performance_table["RMSE_cv_pct"] = (performance_table["RMSE_std"] / performance_table["RMSE_mean"]) * 100.0
performance_table["R2_cv_pct"] = (performance_table["R2_std"].abs() / performance_table["R2_mean"].abs().replace(0, np.nan)) * 100.0

# 2) Grid-search breadth table (shows how much parameter space was explored for each model in each segment).
grid_coverage = (
    segmentwise_tuning_logs
    .groupby(["Segment", "Model"], as_index=False)
    .agg(
        Param_Combinations_Tried=("params", "count"),
        Grid_Best_CV_RMSE=("mean_test_RMSE", "min"),
        Grid_Worst_CV_RMSE=("mean_test_RMSE", "max"),
        Grid_CV_RMSE_STD_Avg=("std_test_score", "mean"),
    )
)

# 3) Final findings table: performance + tuning breadth in one place.
all_findings_table = (
    performance_table
    .merge(grid_coverage, on=["Segment", "Model"], how="left")
    .sort_values(["Segment", "RMSE_mean", "R2_mean"], ascending=[True, True, False])
    .reset_index(drop=True)
)

# 4) Reader-friendly combined snapshot for quick comparison.
segment_model_performance = all_findings_table[[
    "Segment",
    "Model",
    "RMSE_mean",
    "R2_mean",
    "RMSE_rank_in_segment",
    "R2_rank_in_segment",
    "RMSE_cv_pct",
    "R2_cv_pct",
    "Param_Combinations_Tried",
    "Grid_Best_CV_RMSE",
    "Grid_Worst_CV_RMSE",
    "Grid_CV_RMSE_STD_Avg",
]].copy()

print("All Findings Table (one row per segment and model):")
display(all_findings_table.round(4))

print("\nCombined RMSE/R2 performance table:")
display(segment_model_performance.round(4))

# Optional export so the same tables can go into report writing.
all_findings_table.to_csv("../reports/tables/segmentwise_all_findings_table.csv", index=False)
segment_model_performance.to_csv("../reports/tables/segmentwise_rmse_r2_combined.csv", index=False)

print("\nSaved:")
print("- ../reports/tables/segmentwise_all_findings_table.csv")
print("- ../reports/tables/segmentwise_rmse_r2_combined.csv")

Loaded segmentwise_tuned_summary from: ../reports/tables/segmentwise_hyperparam_summary_all_models.csv
Loaded segmentwise_tuning_logs from: ../reports/tables/segmentwise_hyperparam_gridsearch_all_models.csv
All Findings Table (one row per segment and model):


,Segment,Model,RMSE_mean,RMSE_std,MAE_mean,MAE_std,MAPE_mean,MAPE_std,R2_mean,R2_std,Selected_Hyperparameters,RMSE_rank_in_segment,R2_rank_in_segment,RMSE_cv_pct,R2_cv_pct,Param_Combinations_Tried,Grid_Best_CV_RMSE,Grid_Worst_CV_RMSE,Grid_CV_RMSE_STD_Avg
0,dust,XGBoost,155.8868,20.5053,108.8595,13.5598,12.9787,1.9396,0.4273,0.1787,"{""model__learning_rate"": 0.05, ""model__max_dep...",1.0,1.0,13.1540,41.8120,36,146.8835,157.5197,14.7398
1,dust,Random Forest,158.3003,15.1429,109.5878,10.9465,12.8837,1.4685,0.4131,0.1541,"{""model__max_depth"": 16, ""model__min_samples_l...",2.0,2.0,9.5660,37.3137,18,143.5433,149.3316,19.1423
2,dust,LightGBM,161.1379,20.8432,115.5919,16.5197,13.6848,2.1962,0.3913,0.1763,"{""model__learning_rate"": 0.03, ""model__n_estim...",3.0,3.0,12.9350,45.0588,24,152.4689,156.8628,18.1650
3,dust,Gradient Boosting,166.3600,21.6791,117.2099,10.0930,13.9177,1.6818,0.3513,0.1854,"{""model__learning_rate"": 0.03, ""model__max_dep...",4.0,4.0,13.0314,52.7838,27,151.0829,166.9194,16.9520
4,high_grown,Random Forest,225.0651,29.1555,155.6405,23.3280,18.9935,3.4999,0.2867,0.0735,"{""model__max_depth"": 8, ""model__min_samples_le...",1.0,1.0,12.9542,25.6268,18,227.1957,229.2599,41.4730
5,high_grown,LightGBM,227.7371,33.3049,161.4617,25.9648,19.6037,3.7562,0.2737,0.0459,"{""model__learning_rate"": 0.03, ""model__n_estim...",2.0,2.0,14.6243,16.7604,24,230.6975,247.3902,36.9605
6,high_grown,Gradient Boosting,232.3843,35.0363,161.4406,27.6186,19.9873,4.4777,0.2447,0.0487,"{""model__learning_rate"": 0.03, ""model__max_dep...",3.0,3.0,15.0769,19.8971,27,232.9253,254.8710,41.9172
7,high_grown,XGBoost,232.7533,32.6161,162.5668,28.3989,19.8182,3.9574,0.2402,0.0579,"{""model__learning_rate"": 0.03, ""model__max_dep...",4.0,4.0,14.0132,24.0848,36,232.7260,248.2488,41.7136
8,low_grown,Gradient Boosting,263.3410,54.3416,140.1033,19.0782,8.1742,0.8848,0.8760,0.0503,"{""model__learning_rate"": 0.05, ""model__max_dep...",1.0,1.0,20.6354,5.7393,27,266.8086,414.3764,37.7445
9,low_grown,Random Forest,283.1922,80.7930,154.1096,32.4372,8.9281,1.6093,0.8496,0.0897,"{""model__max_depth"": 16, ""model__min_samples_l...",2.0,2.0,28.5294,10.5544,18,290.4873,339.1528,34.0012



RMSE Matrix (lower is better):


Model,Gradient Boosting,LightGBM,Random Forest,XGBoost
Segment,,,,
dust,166.36,161.14,158.30,155.89
high_grown,232.38,227.74,225.07,232.75
low_grown,263.34,331.82,283.19,408.81
off_grade,156.53,150.42,145.11,147.05



R2 Matrix (higher is better):


Model,Gradient Boosting,LightGBM,Random Forest,XGBoost
Segment,,,,
dust,0.3513,0.3913,0.4131,0.4273
high_grown,0.2447,0.2737,0.2867,0.2402
low_grown,0.8760,0.7718,0.8496,0.6594
off_grade,0.1556,0.2329,0.2841,0.2623



Saved:
- ../reports/tables/segmentwise_all_findings_table.csv
- ../reports/tables/segmentwise_rmse_matrix.csv
- ../reports/tables/segmentwise_r2_matrix.csv
